# PrahaarShield – FINAL training (Colab T4, ~25–35 min) · financial scams only · 6 types · EN / Hinglish / Hindi / Marathi
What this run does: MuRIL multi-task, **2 seeds** (the better one on VALIDATION is kept), temperature calibration, two operating points
(**balanced** = best macro-F1 with ≤5% false alarms on validation; **protective** = catches ≥95% of scams on validation if false alarms stay ≤8%),
results per language and **per data source**, and a **with / without outside data** comparison (Ridham115 audited set).
The conversation model (M7) is not trained here: the evaluation showed the rule-based Brain is better, so the app ships the rules.

Drive (`MyDrive/PrahaarShield/`): `prepared_v3/` (train/val/test.jsonl) · `code/` (engine/, common/) · output → `models_final/`.
Runtime → **T4 GPU** → Run all → copy the **SUMMARY** output to Claude.

In [ ]:
!nvidia-smi -L
!pip -q install scikit-learn   # transformers is pre-installed on Colab (pinning an old version fails to build)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/PrahaarShield'
import os, sys
sys.path.append(f'{ROOT}/code')
OUT = f'{ROOT}/models_final'; os.makedirs(OUT, exist_ok=True)
print(os.listdir(f'{ROOT}/prepared_v3'), os.listdir(f'{ROOT}/conversations'))

In [ ]:
# ---------------- CONFIG ----------------
MULTITASK = True
CFG = dict(model='google/muril-base-cased', max_len=256, epochs=8, patience=2, lr_enc=3e-5, lr_head=1e-3, batch=16,
           warmup=0.1, seed=2026, placeholder_dropout=0.3, max_fpr=0.05, seeds=[2026, 7], recall_target=0.95, max_fpr_protective=0.08, ablation_no_ext=True, w_scam=1.0, w_type=1.0, w_pillar=1.0, w_lang=0.2)
TAG = 'muril_multitask' if MULTITASK else 'muril_binary'
import json, random, re, time, csv, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print(DEVICE, TAG)
SUMMARY = {}

In [ ]:
# ---------------- DATA ----------------
load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
train, val, test = [load(f'{ROOT}/prepared_v3/{s}.jsonl') for s in ('train', 'val', 'test')]
try:
    from engine.entities import prepare_for_model
    def to_model_text(t):
        n, e = prepare_for_model(t); return e.text_model
    print('engine found: normalizer + placeholders ON')
except Exception as ex:
    raise SystemExit('Upload code/engine and code/common to Drive first: ' + str(ex))
for rows in (train, val, test):
    for r in rows: r['text_model'] = to_model_text(r['text'])
PILLARS = ['P_IDENTITY','P_SITUATION','P_STAKES','P_CRED','P_RESOURCE','P_CONTROL']
TYPES = sorted({r['scam_type'] for r in train + val + test if r['is_scam'] == 1} - {'other_scam'})   # other_scam = scam without a type label
LANGS = sorted({r['language'] for r in train + val + test})
T2I = {t: i for i, t in enumerate(TYPES)}; L2I = {l: i for i, l in enumerate(LANGS)}
print(len(train), len(val), len(test), TYPES, LANGS)
g = np.mean([r['is_scam'] for r in train]); LW = {}
for l in LANGS:
    p = np.mean([r['is_scam'] for r in train if r['language'] == l] or [g]); p = min(max(p, .02), .98)
    LW[(l, 1)] = float(np.clip(g / p, .5, 3)); LW[(l, 0)] = float(np.clip((1 - g) / (1 - p), .5, 3))
CW = {1: len(train) / (2 * sum(r['is_scam'] for r in train)), 0: len(train) / (2 * sum(1 - r['is_scam'] for r in train))}
tc = np.array([sum(r['scam_type'] == t for r in train) for t in TYPES], dtype=float)
TYPE_W = torch.tensor(tc.sum() / (len(TYPES) * np.maximum(tc, 1)), dtype=torch.float)
known = [r for r in train if r.get('pillar_labels') is not None]
pc = np.array([sum(p in r['pillar_labels'] for r in known) for p in PILLARS], dtype=float)
PIL_POSW = torch.tensor(np.clip((len(known) - pc) / np.maximum(pc, 1), 1, 5), dtype=torch.float)
print('rows with unknown pillars (masked):', len(train) - len(known))

In [ ]:
# ---------------- DATASET ----------------
tok = AutoTokenizer.from_pretrained(CFG['model'])
PH = re.compile(r'<(URL|UPI_LINK|UPI|EMAIL|PHONE|PHONE_INTL|AMOUNT|ACCOUNT|AADHAAR|PAN|CODE)>')
class DS(Dataset):
    def __init__(self, rows, train=False): self.rows, self.train = rows, train
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]; t = r['text_model']
        if self.train and CFG['placeholder_dropout'] > 0:
            t = PH.sub(lambda m: '' if random.random() < CFG['placeholder_dropout'] else m.group(0), t)
        return t, r
def collate(batch):
    texts, rows = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=CFG['max_len'], padding=True, return_tensors='pt')
    y = torch.tensor([r['is_scam'] for r in rows])
    yt = torch.tensor([T2I[r['scam_type']] if r['is_scam'] == 1 and r['scam_type'] in T2I else -100 for r in rows])
    yp = torch.tensor([[1. if p in (r.get('pillar_labels') or []) else 0. for p in PILLARS] for r in rows])
    pm = torch.tensor([0. if r.get('pillar_labels') is None else 1. for r in rows])
    yl = torch.tensor([L2I[r['language']] for r in rows])
    w = torch.tensor([LW[(r['language'], r['is_scam'])] * CW[r['is_scam']] for r in rows], dtype=torch.float)
    return enc, y, yt, yp, pm, yl, w
dl = lambda rows, tr: DataLoader(DS(rows, tr), batch_size=CFG['batch'], shuffle=tr, collate_fn=collate)

In [ ]:
# ---------------- M1 MODEL (same structure as app/engine/text_engine.py) ----------------
class PrahaarModel(nn.Module):
    def __init__(self, name, n_types, n_langs):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name); h = self.enc.config.hidden_size
        self.drop = nn.Dropout(0.1)
        self.scam, self.stype = nn.Linear(h, 2), nn.Linear(h, n_types)
        self.pil, self.lang = nn.Linear(h, len(PILLARS)), nn.Linear(h, n_langs)
    def pooled(self, ids, mask):
        hs = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).float(); return (hs * m).sum(1) / m.sum(1).clamp(min=1)
    def forward(self, ids, mask, return_vec=False):
        v = self.pooled(ids, mask); x = self.drop(v)
        out = (self.scam(x), self.stype(x), self.pil(x), self.lang(x))
        return out + (v,) if return_vec else out
model = PrahaarModel(CFG['model'], len(TYPES), len(LANGS)).to(DEVICE)
ce = nn.CrossEntropyLoss(reduction='none'); ce_type = nn.CrossEntropyLoss(weight=TYPE_W.to(DEVICE), ignore_index=-100)
bce = nn.BCEWithLogitsLoss(pos_weight=PIL_POSW.to(DEVICE), reduction='none'); ce_lang = nn.CrossEntropyLoss()

@torch.no_grad()
def logits_of(rows, m=None):
    mm = m if m is not None else model
    mm.eval(); S, Ty, P, V = [], [], [], []
    for enc, *_ in dl(rows, False):
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            o_s, o_t, o_p, _, v = mm(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE), return_vec=True)
        S.append(o_s.float().cpu()); Ty.append(o_t.float().cpu()); P.append(o_p.float().cpu()); V.append(v.float().cpu())
    return torch.cat(S), torch.cat(Ty), torch.cat(P), torch.cat(V)

In [ ]:
# ---------------- TRAIN M1: 2 seeds, keep the best on VALIDATION ----------------
def type_f1(rows, t):
    idx = [k for k, r in enumerate(rows) if r['is_scam'] == 1 and r['scam_type'] in T2I]
    return f1_score([T2I[rows[k]['scam_type']] for k in idx], t[idx], average='macro') if idx else 0.0
def train_once(seed, rows_tr, tag):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    m = PrahaarModel(CFG['model'], len(TYPES), len(LANGS)).to(DEVICE)
    tr_dl = dl(rows_tr, True)
    enc_p = list(m.enc.parameters()); head_p = [p for n, p in m.named_parameters() if not n.startswith('enc.')]
    opt = torch.optim.AdamW([{'params': enc_p, 'lr': CFG['lr_enc']}, {'params': head_p, 'lr': CFG['lr_head']}], weight_decay=0.01)
    steps = len(tr_dl) * CFG['epochs']; sch = get_linear_schedule_with_warmup(opt, int(steps * CFG['warmup']), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda'); best, bad = -1, 0
    for ep in range(CFG['epochs']):
        m.train(); t0 = time.time(); tot = 0
        for enc, y, yt, yp, pm, yl, w in tr_dl:
            ids, mask = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)
            y, yt, yp, pm, yl, w = y.to(DEVICE), yt.to(DEVICE), yp.to(DEVICE), pm.to(DEVICE), yl.to(DEVICE), w.to(DEVICE)
            with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
                o_s, o_t, o_p, o_l = m(ids, mask)
            loss = CFG['w_scam'] * (ce(o_s.float(), y) * w).mean()
            if (yt != -100).any(): loss = loss + CFG['w_type'] * ce_type(o_t.float(), yt)
            if pm.sum() > 0: loss = loss + CFG['w_pillar'] * (bce(o_p.float(), yp).mean(1) * pm).sum() / pm.sum()
            loss = loss + CFG['w_lang'] * ce_lang(o_l.float(), yl)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
        S, Ty, _, _ = logits_of(val, m); p = torch.softmax(S, -1)[:, 1].numpy()
        f1 = f1_score([r['is_scam'] for r in val], (p >= .5).astype(int), average='macro'); tf1 = type_f1(val, Ty.argmax(-1).numpy())
        score = f1 + 0.5 * tf1
        print(f'  [{tag}] epoch {ep+1}: loss {tot/len(tr_dl):.4f} | val scam F1 {f1:.4f} | val type F1 {tf1:.4f} | {time.time()-t0:.0f}s')
        if score > best: best, bad = score, 0; torch.save(m.state_dict(), f'{OUT}/{tag}.pt')
        else:
            bad += 1
            if bad >= CFG['patience']: print('  early stop'); break
    m.load_state_dict(torch.load(f'{OUT}/{tag}.pt')); return m, best
def quick_eval(m):                     # threshold tuned on VAL (<=5% false alarms), then TEST once
    Sv, _, _, _ = logits_of(val, m); pv = torch.softmax(Sv, -1)[:, 1].numpy(); yv = np.array([r['is_scam'] for r in val])
    th = max([t for t in np.arange(.05, .96, .01) if ((pv >= t)[yv == 0]).mean() <= CFG['max_fpr']] or [.5],
             key=lambda t: f1_score(yv, (pv >= t).astype(int), average='macro'))
    St, _, _, _ = logits_of(test, m); pt = torch.softmax(St, -1)[:, 1].numpy(); yt = np.array([r['is_scam'] for r in test]); pr = (pt >= th).astype(int)
    return dict(threshold=round(float(th), 2), macro_f1=round(f1_score(yt, pr, average='macro'), 4), scam_f1=round(f1_score(yt, pr), 4),
                recall=round(float(pr[yt == 1].mean()), 4), false_alarm=round(float(pr[yt == 0].mean()), 4))
try: del model
except NameError: pass
torch.cuda.empty_cache()
runs, model, best_val = [], None, -1
for sd in CFG['seeds']:
    print('=== seed', sd); m, sc = train_once(sd, train, f'{TAG}_seed{sd}'); q = quick_eval(m); q.update(seed=sd, val_score=round(sc, 4)); runs.append(q); print('  ', q)
    if sc > best_val:
        if model is not None: del model
        model, best_val = m, sc
    else: del m
    torch.cuda.empty_cache()
SUMMARY['seed_runs'] = runs
SUMMARY['seed_spread'] = {k: f"{np.mean([r[k] for r in runs]):.3f} ± {np.std([r[k] for r in runs]):.3f}" for k in ('scam_f1', 'recall', 'false_alarm', 'macro_f1')}
torch.save(model.state_dict(), f'{OUT}/{TAG}.pt'); tok.save_pretrained(f'{OUT}/{TAG}_tokenizer')
print('kept seed with best VALIDATION score:', max(runs, key=lambda r: r['val_score'])['seed'])

In [ ]:
# ---------------- CALIBRATION (temperature scaling on VAL) + threshold ----------------
def fit_T(logits, target, kind):
    T = torch.ones(1, requires_grad=True); o = torch.optim.LBFGS([T], lr=0.05, max_iter=200)
    def step():
        o.zero_grad()
        if kind == 'bce': l = nn.functional.binary_cross_entropy_with_logits(logits / T.clamp(min=0.05), target)
        else: l = nn.functional.cross_entropy(logits / T.clamp(min=0.05), target)
        l.backward(); return l
    o.step(step); return float(T.clamp(min=0.05))
def ece(p, y, bins=10):
    p, y = np.asarray(p), np.asarray(y); e = 0
    for b in range(bins):
        m = (p >= b / bins) & (p < (b + 1) / bins) if b < bins - 1 else (p >= b / bins)
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(e)
Sv, Tyv, Pv, _ = logits_of(val); yv = torch.tensor([r['is_scam'] for r in val])
TEMPS = {'scam': fit_T(Sv, yv, 'ce')}
if MULTITASK:
    idx = [k for k, r in enumerate(val) if r['is_scam'] == 1 and r['scam_type'] in T2I]
    TEMPS['type'] = fit_T(Tyv[idx], torch.tensor([T2I[val[k]['scam_type']] for k in idx]), 'ce')
    km = [k for k, r in enumerate(val) if r.get('pillar_labels') is not None]
    TEMPS['pillar'] = fit_T(Pv[km], torch.tensor([[1. if p in val[k]['pillar_labels'] else 0. for p in PILLARS] for k in km]), 'bce')
pv_raw = torch.softmax(Sv, -1)[:, 1].numpy(); pv = torch.softmax(Sv / TEMPS['scam'], -1)[:, 1].numpy(); yvn = yv.numpy()
print('temperatures', TEMPS, '| ECE val before', round(ece(pv_raw, yvn), 4), 'after', round(ece(pv, yvn), 4))
best_th, best_f = 0.5, -1
for th in np.arange(0.05, 0.96, 0.01):
    pr = (pv >= th).astype(int); fpr = pr[yvn == 0].mean(); f = f1_score(yvn, pr, average='macro')
    if fpr <= CFG['max_fpr'] and f > best_f: best_th, best_f = float(th), f
print(f'balanced threshold {best_th:.2f} (val macro-F1 {best_f:.3f})')
rec_ok = [t for t in np.arange(0.05, 0.96, 0.01) if (pv >= t)[yvn == 1].mean() >= CFG['recall_target']]
prot_th = float(max(rec_ok)) if rec_ok else 0.30
prot_fpr = float((pv >= prot_th)[yvn == 0].mean())
final_th = prot_th if prot_fpr <= CFG['max_fpr_protective'] else best_th
print(f'protective threshold {prot_th:.2f} (val false alarms {prot_fpr:.3f}) -> app uses {final_th:.2f}')
SUMMARY['thresholds'] = dict(balanced=round(best_th, 2), protective=round(prot_th, 2), protective_val_false_alarm=round(prot_fpr, 3), app_uses=round(final_th, 2))
json.dump(dict(CFG=CFG, TYPES=TYPES, LANGS=LANGS, PILLARS=PILLARS, multitask=MULTITASK, threshold=final_th, thresholds=SUMMARY['thresholds'], temperatures=TEMPS, version='final'),
          open(f'{OUT}/{TAG}_config.json', 'w'), indent=2)
SUMMARY['M1_temperatures'] = TEMPS; SUMMARY['M1_threshold'] = final_th

In [ ]:
# ---------------- EVALUATE M1 on TEST ----------------
S, Ty, P, _ = logits_of(test); y = np.array([r['is_scam'] for r in test])
p = torch.softmax(S / TEMPS['scam'], -1)[:, 1].numpy(); pred = (p >= final_th).astype(int); ben = y == 0
print(classification_report(y, pred, target_names=['benign', 'scam'], digits=3))
SUMMARY['M1_test'] = dict(macro_f1=round(f1_score(y, pred, average='macro'), 4), scam_f1=round(f1_score(y, pred), 4),
                          recall=round(float(pred[~ben].mean()), 4), false_alarm=round(float(pred[ben].mean()), 4), ece=round(ece(p, y), 4))
SUMMARY['M1_per_language'] = {l: round(f1_score(y[i], pred[i], average='macro', zero_division=0), 3)
                              for l in LANGS for i in [[k for k, r in enumerate(test) if r['language'] == l]] if len(i) >= 10}
if MULTITASK:
    t = Ty.argmax(-1).numpy(); SUMMARY['M1_type_macro_f1'] = round(type_f1(test, t), 4)
    km = [k for k, r in enumerate(test) if r.get('pillar_labels') is not None]
    yp = np.array([[1 if q in test[k]['pillar_labels'] else 0 for q in PILLARS] for k in km]); pp = (torch.sigmoid(P[km] / TEMPS['pillar']).numpy() >= .5).astype(int)
    SUMMARY['M1_pillar_f1'] = {q: round(f1_score(yp[:, j], pp[:, j], zero_division=0), 3) for j, q in enumerate(PILLARS)}
with open(f'{OUT}/{TAG}_test_errors.csv', 'w', newline='', encoding='utf-8') as fh:
    w = csv.writer(fh); w.writerow(['text', 'true', 'p_scam', 'scam_type', 'pred_type', 'language', 'category'])
    for k, r in enumerate(test):
        wrong = pred[k] != y[k] or (MULTITASK and r['is_scam'] == 1 and r['scam_type'] in T2I and TYPES[Ty[k].argmax()] != r['scam_type'])
        if wrong: w.writerow([r['text'], r['is_scam'], round(float(p[k]), 3), r['scam_type'], TYPES[int(Ty[k].argmax())], r['language'], r['category']])
def mets(th, idx=None):
    idx = np.arange(len(test)) if idx is None else np.array(idx)
    yy, pp = y[idx], (p[idx] >= th).astype(int)
    return dict(n=int(len(idx)), scams=int(yy.sum()), macro_f1=round(f1_score(yy, pp, average='macro', zero_division=0), 4), scam_f1=round(f1_score(yy, pp, zero_division=0), 4),
                precision=round(float(pp[yy == 1].sum() / max(1, pp.sum())), 4), recall=round(float(pp[yy == 1].mean()) if yy.sum() else float('nan'), 4),
                false_alarm=round(float(pp[yy == 0].mean()) if (yy == 0).any() else float('nan'), 4))
SUMMARY['M1_test_balanced'] = mets(SUMMARY['thresholds']['balanced']); SUMMARY['M1_test_protective'] = mets(SUMMARY['thresholds']['protective'])
src = lambda r: 'outside (Ridham115)' if str(r.get('category', '')).startswith('ext_') else ('threat rows v3.2' if str(r.get('category', '')).endswith('_v32') else 'our original data')
SUMMARY['M1_per_source'] = {s_: mets(final_th, [k for k, r in enumerate(test) if src(r) == s_]) for s_ in sorted({src(r) for r in test})}
print(json.dumps(SUMMARY, indent=1))

In [ ]:
# ---------------- ABLATION: does the outside data help? (train WITHOUT it, test on the same test rows) ----------------
is_ext = lambda r: str(r.get('category', '')).startswith('ext_')
if CFG['ablation_no_ext'] and any(is_ext(r) for r in train):
    m0, _ = train_once(CFG['seeds'][0], [r for r in train if not is_ext(r)], f'{TAG}_noext')
    def at05(m, rows):
        S_, _, _, _ = logits_of(rows, m); pr = (torch.softmax(S_, -1)[:, 1].numpy() >= .5).astype(int); yy = np.array([r['is_scam'] for r in rows])
        return dict(n=len(rows), macro_f1=round(f1_score(yy, pr, average='macro', zero_division=0), 4), recall=round(float(pr[yy == 1].mean()), 4), false_alarm=round(float(pr[yy == 0].mean()), 4))
    ext_t, own_t = [r for r in test if is_ext(r)], [r for r in test if not is_ext(r)]
    SUMMARY['ABLATION_outside_data'] = {'with_outside_data': {'outside_test': at05(model, ext_t), 'our_test': at05(model, own_t)},
                                        'without_outside_data': {'outside_test': at05(m0, ext_t), 'our_test': at05(m0, own_t)}}
    del m0; torch.cuda.empty_cache()
    print(json.dumps(SUMMARY['ABLATION_outside_data'], indent=1))
else:
    print('no outside rows in train (run fetch_external.py before prepare_v3.py) -> ablation skipped')

In [ ]:
# ---------------- SUMMARY (copy everything below to Claude) ----------------
json.dump(SUMMARY, open(f'{OUT}/summary_final.json', 'w'), indent=2)
print(json.dumps(SUMMARY, indent=2))
import os; print('\nFiles in', OUT, ':', sorted(f for f in os.listdir(OUT) if not f.endswith('_seed2026.pt') and not f.endswith('_seed7.pt') and not f.endswith('_noext.pt')))